In [ ]:
import pandas as pd

delays = pd.read_csv("../data/delay_history.csv",
                     dtype={"train_no": str},
                     parse_dates=["snapshot_time"])
trains = pd.read_csv("../railpull/data/out/trains.csv", dtype=str)

print(delays.shape, trains.shape)

In [ ]:
print(trains.columns.tolist())
trains.head(10)

In [ ]:
print(delays["snapshot_time"].nunique(), "snapshots so far")

late = delays[delays["cancelled"] == 0]
print(late["delay_min"].describe())

In [ ]:
trains["train_no"] = trains["number"].str.zfill(5)   # pad to 5 digits: "5586" -> "05586"

df = delays.merge(trains, on="train_no", how="left")

print(df.shape)
print("rows with no timetable match:", df["name"].isna().mean())

In [ ]:
df[df["name"].isna()][["train_no", "delay_min", "cancelled"]]

In [ ]:
print(delays["snapshot_time"].nunique())

In [ ]:
df[df["name"].isna()]["train_no"].head(10)

In [ ]:
df["distance_km"] = pd.to_numeric(df["distance_km"], errors="coerce")
df["num_stops"] = pd.to_numeric(df["num_stops"], errors="coerce")

# "26:50" -> 26*60 + 50 = 1610 minutes
tt = df["travel_time"].str.split(":", expand=True)
df["travel_min"] = pd.to_numeric(tt[0], errors="coerce") * 60 + pd.to_numeric(tt[1], errors="coerce")

df[["distance_km", "num_stops", "travel_min"]].describe()



In [ ]:
speed = df["distance_km"] / (df["travel_min"] / 60)
speed.describe()

In [ ]:
late = df[df["cancelled"] == 0]
late.groupby("type_label")["delay_min"].agg(["count", "median", "max"]).sort_values("count", ascending=False)

In [ ]:
print(delays["snapshot_time"].nunique())

In [ ]:
import pickle

with open("../artifacts/preprocessor.pkl", "rb") as f:
    preprocessor = pickle.load(f)

preprocessor

In [ ]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

raw = pd.read_csv("../artifacts/raw.csv", dtype={"train_no": str})

from src.Components.data_transformation import add_features, DataTransformation, NUMERIC_COLS, CATEGORICAL_COLS

raw = add_features(raw)
X = raw[NUMERIC_COLS + CATEGORICAL_COLS]
y = np.log1p(raw["delay_min"])

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr, te = next(gss.split(X, y, groups=raw["train_no"]))

pre = DataTransformation().get_preprocessor()
Xtr = pre.fit_transform(X.iloc[tr])
Xte = pre.transform(X.iloc[te])

rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42, n_jobs=-1)
rf.fit(Xtr, y.iloc[tr])
pred = np.expm1(rf.predict(Xte))
print("MAE on unseen trains:", mean_absolute_error(raw["delay_min"].iloc[te], pred))

baseline = np.expm1(np.median(y.iloc[tr]))
print("Baseline MAE on the same unseen trains:",
      mean_absolute_error(raw["delay_min"].iloc[te], [baseline] * len(te)))

In [ ]:
import pandas as pd
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("../artifacts/raw.csv", dtype={"train_no": str},
                parse_dates=["snapshot_time"])
d = d.sort_values(["train_no", "snapshot_time"])
d["prev_delay"] = d.groupby("train_no")["delay_min"].shift(1)

p = d.dropna(subset=["prev_delay"])
print(len(p), "rows that have a previous snapshot")
print("Persistence MAE:", mean_absolute_error(p["delay_min"], p["prev_delay"]))
print("Baseline MAE:   ", mean_absolute_error(
    p["delay_min"], [p["delay_min"].median()] * len(p)))

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from src.Components.data_transformation import count_run_days

d = pd.read_csv("../artifacts/raw.csv", dtype={"train_no": str},
                parse_dates=["snapshot_time"])
d = d.sort_values(["train_no", "snapshot_time"])

g = d.groupby("train_no")
d["prev_delay"] = g["delay_min"].shift(1)
d["gap_min"] = g["snapshot_time"].diff().dt.total_seconds() / 60
d["n_run_days"] = d["runs_days"].apply(count_run_days)

p = d.dropna(subset=["prev_delay", "gap_min"]).copy()
p["delta"] = p["delay_min"] - p["prev_delay"]

features = ["prev_delay", "gap_min", "distance_km", "num_stops",
            "travel_min", "hour", "n_run_days"]
X = pd.concat([p[features], pd.get_dummies(p["type_label"])], axis=1)

last = p["snapshot_time"].max()
test = p["snapshot_time"] == last
print("train pairs:", (~test).sum(), "| test pairs:", test.sum())

model = GradientBoostingRegressor(random_state=42)
model.fit(X[~test], p.loc[~test, "delta"])

pred_now = p.loc[test, "prev_delay"] + model.predict(X[test])
true_now = p.loc[test, "delay_min"]

print("Persistence MAE:", mean_absolute_error(true_now, p.loc[test, "prev_delay"]))
print("Model MAE:      ", mean_absolute_error(true_now, pred_now))
print("Gap (min), typical:", p["gap_min"].median())

In [ ]:
import pandas as pd

r = pd.read_csv("../artifacts/raw.csv", parse_dates=["snapshot_time"])
print(r.groupby("snapshot_time").agg(
    pairs=("delta", "size"),
    median_gap_min=("gap_min", "median"),
    persistence_mae=("delta", lambda s: s.abs().mean()),
))